# Лабораторная работа №8: Active learning и human-in-the-loop

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Реализовать random, entropy и diversity sampling.
- Провести несколько AL-итераций.
- Построить learning curves при равном бюджете.


## Теоретическая часть

Active learning выбирает следующий batch из pool. Uncertainty ищет неуверенные объекты, diversity покрывает пространство, disagreement использует расхождение моделей. Сравнение корректно только при одинаковом annotation budget.


### Математическая постановка

Entropy sampling: $u(x)=-\sum_c p(c\mid x)\log p(c\mid x)$. Diversity можно реализовать k-center: выбрать объект, максимизирующий расстояние до уже размеченного множества.


### Материалы

- [Active Learning survey](https://arxiv.org/abs/2009.00236)
- [scikit-learn calibration](https://scikit-learn.org/stable/modules/calibration.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Работа выполняется на изображениях рукописных цифр `sklearn.datasets.load_digits`; oracle в этом воспроизводимом эксперименте симулирует человека, открывая истинную метку только после отбора объекта. FAST_DEV_RUN — два раунда проверки кода, полный режим — восемь раундов и повторный random baseline.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))

## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Стратифицированный отложенный тест

*Вес: 1 балл.*

**Постановка.** Этап «стратифицированный отложенный тест» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Разделите изображения цифр по стратифицированному правилу до выбора стратегии.
2. Сохраните и предъявите индексы train/test изображений цифр.

**Результат.** индексы train/test изображений цифр.

**Критерий принятия.** Задание считается принятым, если индексы train/test изображений цифр получены из фактических входов этого этапа и доступны для проверки.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from collections import Counter

digits = load_digits()
X = digits.data.astype(np.float64) / 16.0     # 1797 x 64 признака, пиксели 0..16 -> 0..1
y_all = digits.target.astype(int)
images_all = digits.images                    # 1797 x 8 x 8
all_idx = np.arange(len(y_all))

# Стратифицированное разбиение делаем ДО выбора стратегии; дальше оно не меняется.
train_idx, test_idx = train_test_split(all_idx, test_size=0.3, stratify=y_all, random_state=SEED)
train_idx, test_idx = np.sort(train_idx), np.sort(test_idx)
assert not (set(train_idx) & set(test_idx)) and len(train_idx) + len(test_idx) == len(all_idx)

for name, idx in [('train', train_idx), ('test', test_idx)]:
    print(f'{name:5s} n={len(idx):4d}  по классам: {dict(sorted(Counter(y_all[idx].tolist()).items()))}')
save_json('split_indices.json', {'seed': SEED, 'test_size': 0.3, 'stratify_by': 'digit label',
                                 'train_idx': train_idx.tolist(), 'test_idx': test_idx.tolist()})

train n=1257  по классам: {0: 124, 1: 127, 2: 124, 3: 128, 4: 127, 5: 127, 6: 127, 7: 125, 8: 122, 9: 126}
test  n= 540  по классам: {0: 54, 1: 55, 2: 53, 3: 55, 4: 54, 5: 55, 6: 54, 7: 54, 8: 52, 9: 54}


### Начальная размеченная выборка

*Вес: 1 балл.*

**Постановка.** Этап «начальная размеченная выборка» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Задайте initial с каждым классом и неразмеченный pool только из train.
2. Сохраните и предъявите непересекающиеся initial и pool.

**Результат.** непересекающиеся initial и pool.

**Критерий принятия.** Задание считается принятым, если непересекающиеся initial и pool получены из фактических входов этого этапа и доступны для проверки.

In [3]:
N_PER_CLASS_INIT = 2                           # initial: по 2 примера каждого из 10 классов -> 20 меток

def make_initial(seed, per_class=N_PER_CLASS_INIT):
    r = np.random.RandomState(seed)
    parts = [r.choice(train_idx[y_all[train_idx] == c], per_class, replace=False) for c in range(10)]
    initial = np.sort(np.concatenate(parts))
    return initial, np.setdiff1d(train_idx, initial)     # pool строится ТОЛЬКО из train

initial_idx, pool_idx = make_initial(SEED)
assert not (set(initial_idx) & set(pool_idx))                          # непересекающиеся
assert set(initial_idx) <= set(train_idx) and set(pool_idx) <= set(train_idx)
assert not (set(initial_idx) | set(pool_idx)) & set(test_idx)          # test не затронут
assert set(initial_idx) | set(pool_idx) == set(train_idx)
print('initial:', len(initial_idx), 'pool:', len(pool_idx), 'классы в initial:', sorted(Counter(y_all[initial_idx].tolist()).items()))
save_json('initial_pool.json', {'seed': SEED, 'initial_idx': initial_idx.tolist(), 'pool_idx': pool_idx.tolist()})

initial: 20 pool: 1237 классы в initial: [(0, 2), (1, 2), (2, 2), (3, 2), (4, 2), (5, 2), (6, 2), (7, 2), (8, 2), (9, 2)]


### Протокол симулированного разметчика

*Вес: 1 балл.*

**Постановка.** Этап «протокол симулированного разметчика» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните split и документируйте, что oracle открывает метку лишь после выбора.
2. Сохраните и предъявите artifacts/split.json с происхождением меток.

**Результат.** artifacts/split.json с происхождением меток.

**Критерий принятия.** Задание считается принятым, если artifacts/split.json с происхождением меток получены из фактических входов этого этапа и доступны для проверки.

In [4]:
class Oracle:
    """Симулированный разметчик. Истинные метки хранятся приватно и открываются
    только по явному запросу query(), и только для объектов обучающей части.
    Метки test через oracle получить нельзя (PermissionError)."""
    def __init__(self, labels, allowed_idx):
        self.__labels = np.asarray(labels).copy()
        self.allowed = set(map(int, allowed_idx))
        self.revealed = {}                      # idx -> {label, round, strategy}
    def query(self, indices, round_, strategy):
        out = []
        for i in map(int, indices):
            if i not in self.allowed:
                raise PermissionError(f'oracle: объект {i} вне обучающей части')
            if i in self.revealed:
                raise ValueError(f'oracle: объект {i} уже размечен (бюджет не тратим повторно)')
            lab = int(self.__labels[i])
            self.revealed[i] = {'label': lab, 'round': round_, 'strategy': strategy}
            out.append(lab)
        return np.array(out)
    @property
    def budget(self):
        return len(self.revealed)

# Проверка протокола: метка test недоступна, повторный запрос не тратит бюджет.
_o = Oracle(y_all, train_idx)
try:
    _o.query([int(test_idx[0])], 0, 'check'); raise AssertionError('oracle выдал метку test!')
except PermissionError as e:
    print('OK, test защищён:', e)

split_record = {
    'seed': SEED, 'dataset': 'sklearn.datasets.load_digits (1797 изображений 8x8)',
    'train_idx': train_idx.tolist(), 'test_idx': test_idx.tolist(),
    'initial_idx': initial_idx.tolist(), 'pool_idx': pool_idx.tolist(),
    'label_provenance': {
        'source': 'digits.target (истинные метки набора)',
        'initial': 'открыты oracle.query(initial_idx) в раунде 0, входят в бюджет (20 меток)',
        'queried': 'метка объекта открывается oracle.query() только ПОСЛЕ его выбора стратегией; '
                   'стратегия видит лишь признаки pool и предсказания модели, обученной на уже открытых метках',
        'test': 'через oracle недоступны; y_all[test_idx] используется только в evaluate() для отчёта '
                'и никогда не участвует в выборе объектов или обучении',
        'pool': 'метки pool не читаются до запроса'},
}
save_json('split.json', split_record)
print('split.json сохранён, ключи:', list(split_record))

OK, test защищён: oracle: объект 4 вне обучающей части
split.json сохранён, ключи: ['seed', 'dataset', 'train_idx', 'test_idx', 'initial_idx', 'pool_idx', 'label_provenance']


### Три стратегии запроса

*Вес: 1 балл.*

**Постановка.** Этап «три стратегии запроса» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Реализуйте случайный, энтропийный и k-center выбор; в k-center обновляйте расстояния после каждого объекта.
2. Сохраните и предъявите выбор random, entropy и k-center.

**Результат.** выбор random, entropy и k-center.

**Критерий принятия.** Задание считается принятым, если выбор random, entropy и k-center получены из фактических входов этого этапа и доступны для проверки.

In [5]:
from scipy.spatial.distance import cdist
from sklearn.linear_model import LogisticRegression
import warnings; warnings.filterwarnings('ignore')

def make_model():
    return LogisticRegression(max_iter=2000)    # детерминированный lbfgs

def entropy_scores(proba):
    p = np.clip(proba, 1e-12, 1.0)
    return -(p * np.log(p)).sum(axis=1)         # u(x) = -sum_c p(c|x) log p(c|x)

def select_random(pool, k, rng):
    return rng.choice(pool, size=k, replace=False)

def select_entropy(pool, k, model):
    u = entropy_scores(model.predict_proba(X[pool]))
    top = np.argsort(-u, kind='stable')[:k]     # k самых неопределённых
    return pool[top]

def select_kcenter(pool, k, labeled, return_dist=False):
    """Жадный k-center: каждый раз берём объект, максимально удалённый от размеченного множества.
    Расстояния до ближайшего центра ОБНОВЛЯЮТСЯ после каждого выбранного объекта."""
    Xp = X[pool]
    d = cdist(Xp, X[np.asarray(labeled)]).min(axis=1)
    chosen, dists = [], []
    for _ in range(k):
        j = int(np.argmax(d))
        chosen.append(int(pool[j])); dists.append(float(d[j]))
        d = np.minimum(d, cdist(Xp, Xp[j:j+1])[:, 0])   # обновление после каждого объекта
        d[j] = -1.0                                      # исключаем выбранный
    chosen = np.array(chosen)
    return (chosen, dists) if return_dist else chosen

def select_hybrid(pool, k, labeled, model, factor=5):
    """(для бонуса) кандидаты = top-(factor*k) по энтропии, из них k-center относительно labeled."""
    u = entropy_scores(model.predict_proba(X[pool]))
    cand = pool[np.argsort(-u, kind='stable')[:factor * k]]
    return select_kcenter(cand, k, labeled)

# Демонстрация на модели, обученной только на initial (метки initial получены у oracle).
BATCH = 10
_demo_oracle = Oracle(y_all, train_idx)
_y0 = _demo_oracle.query(initial_idx, 0, 'demo')
_m0 = make_model().fit(X[initial_idx], _y0)
_rng = np.random.RandomState(SEED)
sel_random = select_random(pool_idx, BATCH, _rng)
sel_entropy = select_entropy(pool_idx, BATCH, _m0)
sel_kcenter, kc_d = select_kcenter(pool_idx, BATCH, initial_idx, return_dist=True)
for n, s in [('random', sel_random), ('entropy', sel_entropy), ('k-center', sel_kcenter)]:
    assert len(set(s)) == BATCH and set(s) <= set(pool_idx)
    print(f'{n:9s}', sorted(map(int, s)))
save_json('selections_demo.json', {
    'note': 'первый батч из 10 объектов при модели, обученной на initial (20 меток)',
    'random': list(map(int, sel_random)), 'entropy': list(map(int, sel_entropy)),
    'entropy_scores': entropy_scores(_m0.predict_proba(X[sel_entropy])).tolist(),
    'kcenter': list(map(int, sel_kcenter)), 'kcenter_dist_at_pick': kc_d})

random    [111, 247, 366, 883, 1086, 1254, 1260, 1353, 1409, 1636]
entropy   [53, 255, 296, 414, 424, 426, 500, 630, 829, 1468]
k-center  [8, 947, 951, 985, 1024, 1272, 1400, 1495, 1576, 1595]


### Итерационное переобучение

*Вес: 1 балл.*

**Постановка.** Этап «итерационное переобучение» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. После каждого запроса дообучайте классификатор на открытых метках и фиксируйте trace.
2. Сохраните и предъявите функция run с фактическими запрошенными метками.

**Результат.** функция run с фактическими запрошенными метками.

**Критерий принятия.** Задание считается принятым, если функция run с фактическими запрошенными метками получены из фактических входов этого этапа и доступны для проверки.

In [6]:
from sklearn.metrics import accuracy_score
ROUNDS = 2 if FAST_DEV_RUN else 8               # FAST: два раунда; полный: восемь
STRATEGIES = ('random', 'entropy', 'kcenter')

def evaluate(model, eval_idx=None):
    """Только отчётная метрика: eval-метки не участвуют в выборе объектов."""
    eval_idx = test_idx if eval_idx is None else eval_idx
    return float(accuracy_score(y_all[eval_idx], model.predict(X[eval_idx])))

def batch_stats(idx):
    d = cdist(X[idx], X[idx]); k = len(idx)
    return float(d.sum() / (k * (k - 1))), int(len(set(y_all[idx].tolist())))

def run(strategy, seed, initial=None, pool=None, eval_idx=None, rounds=None, batch=BATCH, hybrid_factor=5):
    """Active learning: после КАЖДОГО запроса классификатор переобучается на всех открытых метках."""
    initial = initial_idx if initial is None else initial
    pool = pool_idx if pool is None else pool
    rounds = ROUNDS if rounds is None else rounds
    rng = np.random.RandomState(seed)
    oracle = Oracle(y_all, train_idx)
    labeled = [int(i) for i in initial]
    y_lab = list(oracle.query(initial, 0, strategy))
    pool = np.array(pool)
    model = make_model().fit(X[labeled], y_lab)
    trace = [{'round': 0, 'n_labels': len(labeled), 'accuracy': evaluate(model, eval_idx),
              'queried_idx': [], 'queried_labels': []}]
    for r in range(1, rounds + 1):
        if strategy == 'random':    q = select_random(pool, batch, rng)
        elif strategy == 'entropy': q = select_entropy(pool, batch, model)
        elif strategy == 'kcenter': q = select_kcenter(pool, batch, labeled)
        elif strategy == 'hybrid':  q = select_hybrid(pool, batch, labeled, model, hybrid_factor)
        else: raise ValueError(strategy)
        q = np.asarray(q, dtype=int)
        assert len(set(q)) == batch and set(q) <= set(pool) and not set(q) & set(labeled)
        mean_d, n_cls = None, None
        labs = oracle.query(q, r, strategy)                       # метки открываются только теперь
        mean_d, n_cls = batch_stats(q)
        labeled += q.tolist(); y_lab += labs.tolist()
        pool = np.setdiff1d(pool, q)
        model = make_model().fit(X[labeled], y_lab)               # переобучение
        trace.append({'round': r, 'n_labels': len(labeled), 'accuracy': evaluate(model, eval_idx),
                      'queried_idx': q.tolist(), 'queried_labels': labs.tolist(),
                      'batch_mean_pairwise_dist': mean_d, 'batch_n_classes': n_cls})
    assert oracle.budget == len(labeled) == len(initial) + rounds * batch
    return {'strategy': strategy, 'seed': seed, 'trace': trace, 'labeled_idx': labeled,
            'labeled_y': y_lab, 'oracle_budget': oracle.budget, 'model': model}

def public(res):                                   # без объекта модели - для JSON
    return {k: v for k, v in res.items() if k != 'model'}

ex = run('entropy', SEED, rounds=2)
for t in ex['trace']:
    print(f"раунд {t['round']}: меток={t['n_labels']:3d}  acc={t['accuracy']:.4f}  запрошено={t['queried_idx']}  метки={t['queried_labels']}")
assert all(y_all[t['queried_idx']].tolist() == t['queried_labels'] for t in ex['trace'][1:])
save_json('run_example.json', public(ex))

раунд 0: меток= 20  acc=0.7185  запрошено=[]  метки=[]
раунд 1: меток= 30  acc=0.5074  запрошено=[1468, 829, 255, 500, 414, 296, 426, 630, 53, 424]  метки=[8, 8, 8, 8, 8, 8, 8, 8, 8, 8]
раунд 2: меток= 40  acc=0.5426  запрошено=[1729, 1595, 1038, 120, 715, 1571, 756, 1394, 1386, 1738]  метки=[3, 7, 9, 5, 9, 8, 4, 1, 1, 5]


### Кривые при общем бюджете

*Вес: 1 балл.*

**Постановка.** Этап «кривые при общем бюджете» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Вычислите точки кривых для одинакового числа запросов и сохраните бюджеты.
2. Сохраните и предъявите artifacts/learning_curves.json.

**Результат.** artifacts/learning_curves.json.

**Критерий принятия.** Задание считается принятым, если artifacts/learning_curves.json получены из фактических входов этого этапа и доступны для проверки.

In [7]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

main_runs = {s: run(s, SEED) for s in STRATEGIES}                 # один и тот же initial/pool/test
curves = {s: {'n_labels': [t['n_labels'] for t in r['trace']],
              'accuracy': [t['accuracy'] for t in r['trace']]} for s, r in main_runs.items()}
budgets = curves['random']['n_labels']
assert all(c['n_labels'] == budgets for c in curves.values())     # одинаковый бюджет у всех стратегий
for s in STRATEGIES:
    save_json(f'run_trace_{s}.json', public(main_runs[s]))

save_json('learning_curves.json', {
    'protocol': {'seed': SEED, 'initial_labels': len(initial_idx), 'batch': BATCH, 'rounds': ROUNDS,
                 'budgets': budgets, 'classifier': 'LogisticRegression(max_iter=2000) на пикселях/16',
                 'test_size': int(len(test_idx))},
    'curves': curves})

print('бюджеты:', budgets)
print(f"{'меток':>6s} " + ' '.join(f'{s:>9s}' for s in STRATEGIES))
for i, b in enumerate(budgets):
    print(f'{b:6d} ' + ' '.join(f"{curves[s]['accuracy'][i]:9.4f}" for s in STRATEGIES))

plt.figure(figsize=(6.5, 4))
for s in STRATEGIES: plt.plot(budgets, curves[s]['accuracy'], marker='o', label=s)
plt.xlabel('число полученных меток'); plt.ylabel('accuracy на отложенном test'); plt.grid(alpha=.3); plt.legend()
plt.title('Learning curves (общий бюджет)'); plt.tight_layout(); plt.savefig(ARTIFACTS/'learning_curves.png', dpi=130); plt.show()

бюджеты: [20, 30, 40, 50, 60, 70, 80, 90, 100]
 меток    random   entropy   kcenter
    20    0.7185    0.7185    0.7185
    30    0.7259    0.5074    0.7667
    40    0.7907    0.5426    0.8463
    50    0.7907    0.6352    0.8630
    60    0.8370    0.7241    0.8944
    70    0.8611    0.7796    0.8981
    80    0.8704    0.8093    0.9019
    90    0.8759    0.8519    0.9074
   100    0.8815    0.8630    0.9185


### Интеграл по бюджету разметки

*Вес: 1 балл.*

**Постановка.** Этап «интеграл по бюджету разметки» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Проинтегрируйте кривые accuracy по числу полученных меток.
2. Сохраните и предъявите AUC и конечная accuracy из фактических кривых.

**Результат.** AUC и конечная accuracy из фактических кривых.

**Критерий принятия.** Задание считается принятым, если AUC и конечная accuracy из фактических кривых получены из фактических входов этого этапа и доступны для проверки.

In [8]:
def trapezoid(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    return float(np.sum((x[1:] - x[:-1]) * (y[1:] + y[:-1]) / 2))

def auc_stats(curve):
    raw = trapezoid(curve['n_labels'], curve['accuracy'])
    span = curve['n_labels'][-1] - curve['n_labels'][0]
    return {'auc_raw': raw, 'auc_norm': raw / span, 'final_accuracy': curve['accuracy'][-1],
            'n_labels_final': curve['n_labels'][-1]}

auc_main = {s: auc_stats(curves[s]) for s in STRATEGIES}
for s, v in auc_main.items():
    print(f"{s:8s} AUC(норм.)={v['auc_norm']:.4f}  AUC(raw)={v['auc_raw']:.2f}  финальная acc={v['final_accuracy']:.4f} при {v['n_labels_final']} метках")
save_json('auc.json', {'method': 'трапеции по числу меток; auc_norm = auc_raw / (N_final - N_initial) = средняя accuracy по бюджету',
                       'values': auc_main})

random   AUC(норм.)=0.8190  AUC(raw)=65.52  финальная acc=0.8815 при 100 метках
entropy  AUC(норм.)=0.7051  AUC(raw)=56.41  финальная acc=0.8630 при 100 метках
kcenter  AUC(норм.)=0.8620  AUC(raw)=68.96  финальная acc=0.9185 при 100 метках


### Повторные случайные запросы

*Вес: 1 балл.*

**Постановка.** Этап «повторные случайные запросы» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Повторите random с разными генераторами при неизменном train/test.
2. Сохраните и предъявите результаты random при различных seed.

**Результат.** результаты random при различных seed.

**Критерий принятия.** Задание считается принятым, если результаты random при различных seed получены из фактических входов этого этапа и доступны для проверки.

In [9]:
N_REP = 3 if FAST_DEV_RUN else 10
rep_seeds = [SEED + i for i in range(N_REP)]

# (а) random с разными генераторами; train/test и initial неизменны
rand_runs = [run('random', s) for s in rep_seeds]
rand_stats = [{'seed': s, **auc_stats({'n_labels': [t['n_labels'] for t in r['trace']],
                                       'accuracy': [t['accuracy'] for t in r['trace']]}),
               'accuracy_curve': [t['accuracy'] for t in r['trace']]} for s, r in zip(rep_seeds, rand_runs)]
assert rand_stats[0]['accuracy_curve'] == curves['random']['accuracy']      # воспроизводимость seed=SEED
au = np.array([r['auc_norm'] for r in rand_stats]); fi = np.array([r['final_accuracy'] for r in rand_stats])
random_summary = {'n_repeats': N_REP, 'seeds': rep_seeds, 'auc_norm_mean': float(au.mean()), 'auc_norm_std': float(au.std(ddof=1)),
                  'final_acc_mean': float(fi.mean()), 'final_acc_std': float(fi.std(ddof=1)),
                  'auc_norm_min': float(au.min()), 'auc_norm_max': float(au.max())}
print(f"random x{N_REP}: AUC={au.mean():.4f}±{au.std(ddof=1):.4f} [{au.min():.4f}; {au.max():.4f}], final={fi.mean():.4f}±{fi.std(ddof=1):.4f}")
save_json('random_seeds.json', {'summary': random_summary, 'runs': rand_stats})

# (б) устойчивость выводов к выбору initial: тот же test, все три стратегии, разные начальные 20 меток
multi = {s: [] for s in STRATEGIES}
for s0 in rep_seeds:
    ini, pl = make_initial(s0)
    for s in STRATEGIES:
        r = run(s, s0, initial=ini, pool=pl)
        multi[s].append(auc_stats({'n_labels': [t['n_labels'] for t in r['trace']],
                                   'accuracy': [t['accuracy'] for t in r['trace']]}))
multi_summary = {}
for s in STRATEGIES:
    a = np.array([m['auc_norm'] for m in multi[s]]); f = np.array([m['final_accuracy'] for m in multi[s]])
    multi_summary[s] = {'auc_norm_mean': float(a.mean()), 'auc_norm_std': float(a.std(ddof=1)),
                        'final_acc_mean': float(f.mean()), 'final_acc_std': float(f.std(ddof=1))}
    print(f"[разные initial] {s:8s} AUC={a.mean():.4f}±{a.std(ddof=1):.4f}  final={f.mean():.4f}±{f.std(ddof=1):.4f}")
paired = {s: int(sum(multi[s][i]['auc_norm'] > multi['random'][i]['auc_norm'] for i in range(N_REP))) for s in ('entropy', 'kcenter')}
print('число initial-сидов, где стратегия по AUC лучше random:', paired, 'из', N_REP)
save_json('multi_init.json', {'n_repeats': N_REP, 'seeds': rep_seeds, 'summary': multi_summary,
                              'wins_over_random_by_auc': paired, 'per_seed': multi})

random x10: AUC=0.8157±0.0114 [0.7997; 0.8358], final=0.8757±0.0131
[разные initial] random   AUC=0.8314±0.0243  final=0.8830±0.0195
[разные initial] entropy  AUC=0.7874±0.0350  final=0.8798±0.0206
[разные initial] kcenter  AUC=0.8575±0.0090  final=0.9096±0.0082
число initial-сидов, где стратегия по AUC лучше random: {'entropy': 0, 'kcenter': 10} из 10


### Разбор ошибок отложенного теста

*Вес: 1 балл.*

**Постановка.** Этап «разбор ошибок отложенного теста» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните изображения и истинные/предсказанные классы ошибочных test-примеров.
2. Сохраните и предъявите artifacts/test_errors.json с неверно распознанными изображениями.

**Результат.** artifacts/test_errors.json с неверно распознанными изображениями.

**Критерий принятия.** Задание считается принятым, если artifacts/test_errors.json с неверно распознанными изображениями получены из фактических входов этого этапа и доступны для проверки.

In [10]:
def errors_of(model):
    pred = model.predict(X[test_idx]); wrong = np.where(pred != y_all[test_idx])[0]
    return [{'index': int(test_idx[w]), 'true': int(y_all[test_idx[w]]), 'pred': int(pred[w]),
             'image_8x8': images_all[test_idx[w]].astype(int).tolist()} for w in wrong]

test_errors = {}
for s in STRATEGIES:
    errs = errors_of(main_runs[s]['model'])
    pairs = Counter((e['true'], e['pred']) for e in errs).most_common(5)
    test_errors[s] = {'n_test': int(len(test_idx)), 'n_errors': len(errs), 'error_rate': len(errs) / len(test_idx),
                      'most_common_confusions(true,pred,count)': [[a, b, c] for (a, b), c in pairs], 'errors': errs}
    print(f"{s:8s} ошибок {len(errs):3d}/{len(test_idx)}; частые пары (true,pred): {pairs}")
save_json('test_errors.json', test_errors)

fig, axes = plt.subplots(len(STRATEGIES), 12, figsize=(14, 4.2))
for r, s in enumerate(STRATEGIES):
    errs = test_errors[s]['errors'][:12]
    for c in range(12):
        ax = axes[r, c]; ax.axis('off')
        if c < len(errs):
            ax.imshow(np.array(errs[c]['image_8x8']), cmap='gray_r'); ax.set_title(f"{errs[c]['true']}→{errs[c]['pred']}", fontsize=8)
    axes[r, 0].text(-0.6, 0.5, s, transform=axes[r, 0].transAxes, ha='right', va='center', fontsize=9)
plt.suptitle('Ошибки на test после бюджета меток (истина→предсказание)'); plt.tight_layout()
plt.savefig(ARTIFACTS/'test_errors.png', dpi=130); plt.show()

random   ошибок  64/540; частые пары (true,pred): [((8, 1), 11), ((9, 8), 10), ((1, 8), 9), ((9, 1), 6), ((2, 1), 5)]
entropy  ошибок  74/540; частые пары (true,pred): [((6, 8), 13), ((1, 2), 10), ((2, 8), 9), ((9, 8), 9), ((1, 8), 8)]
kcenter  ошибок  44/540; частые пары (true,pred): [((8, 1), 8), ((9, 8), 5), ((1, 8), 5), ((8, 9), 2), ((0, 4), 2)]


### Сводка сравнительного эксперимента

*Вес: 1 балл.*

**Постановка.** Этап «сводка сравнительного эксперимента» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сопоставьте AUC и конечную точность по трём стратегиям.
2. Сохраните и предъявите artifacts/comparison.json.

**Результат.** artifacts/comparison.json.

**Критерий принятия.** Задание считается принятым, если artifacts/comparison.json получены из фактических входов этого этапа и доступны для проверки.

In [11]:
comparison = {
    'protocol': {'initial_labels': len(initial_idx), 'batch': BATCH, 'rounds': ROUNDS, 'final_budget': budgets[-1],
                 'test_size': int(len(test_idx)), 'classifier': 'LogisticRegression', 'seed': SEED},
    'main_run_single_seed': auc_main,
    'random_over_seeds': random_summary,
    'over_initial_sets': multi_summary,
    'wins_over_random_by_auc_over_initial_sets': paired,
}
comparison['ranking_by_auc_main_run'] = sorted(STRATEGIES, key=lambda s: -auc_main[s]['auc_norm'])
comparison['ranking_by_final_acc_main_run'] = sorted(STRATEGIES, key=lambda s: -auc_main[s]['final_accuracy'])
rs = random_summary
comparison['vs_random_distribution'] = {
    s: {'auc_diff_vs_random_mean': auc_main[s]['auc_norm'] - rs['auc_norm_mean'],
        'in_std_units': (auc_main[s]['auc_norm'] - rs['auc_norm_mean']) / rs['auc_norm_std'],
        'final_diff_vs_random_mean': auc_main[s]['final_accuracy'] - rs['final_acc_mean']}
    for s in ('entropy', 'kcenter')}
save_json('comparison.json', comparison)

print(f"{'стратегия':10s}{'AUC(норм)':>11s}{'финал acc':>11s}")
for s in STRATEGIES: print(f"{s:10s}{auc_main[s]['auc_norm']:11.4f}{auc_main[s]['final_accuracy']:11.4f}")
print(f"{'random(μ)':10s}{rs['auc_norm_mean']:11.4f}{rs['final_acc_mean']:11.4f}   (±{rs['auc_norm_std']:.4f} / ±{rs['final_acc_std']:.4f}, {rs['n_repeats']} seeds)")
print('рейтинг по AUC:', comparison['ranking_by_auc_main_run'], '| по финальной accuracy:', comparison['ranking_by_final_acc_main_run'])
for s, v in comparison['vs_random_distribution'].items():
    print(f"{s}: ΔAUC против среднего random = {v['auc_diff_vs_random_mean']:+.4f} ({v['in_std_units']:+.1f}σ случайного разброса)")

стратегия   AUC(норм)  финал acc
random         0.8190     0.8815
entropy        0.7051     0.8630
kcenter        0.8620     0.9185
random(μ)      0.8157     0.8757   (±0.0114 / ±0.0131, 10 seeds)
рейтинг по AUC: ['kcenter', 'random', 'entropy'] | по финальной accuracy: ['kcenter', 'random', 'entropy']
entropy: ΔAUC против среднего random = -0.1106 (-9.7σ случайного разброса)
kcenter: ΔAUC против среднего random = +0.0463 (+4.1σ случайного разброса)


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [12]:
# Гипотеза (сформулирована до запуска, test НЕ используется):
# H1: batch-режим entropy-сэмплинга выбирает избыточные (похожие) объекты: среднее попарное расстояние
#     внутри батча и число разных классов в батче у entropy ниже, чем у k-center и random.
# H2: гибрид «top-5k по энтропии -> k-center» не хуже entropy по AUC (диверсификация устраняет избыточность).
# Протокол: дополнительные независимые входы = внутренние разбиения ОБУЧАЮЩЕЙ части (70% inner-train / 30% inner-val),
# метрика считается на inner-val, отложенный test не открывается.
N_INNER = 3 if FAST_DEV_RUN else 5
BONUS_STRATS = ('random', 'entropy', 'kcenter', 'hybrid')
bonus = {s: {'auc': [], 'final': [], 'dist': [], 'ncls': []} for s in BONUS_STRATS}
for b in range(N_INNER):
    sd = 1000 + b
    in_tr, in_val = train_test_split(train_idx, test_size=0.3, stratify=y_all[train_idx], random_state=sd)
    r_ = np.random.RandomState(sd)
    ini = np.sort(np.concatenate([r_.choice(in_tr[y_all[in_tr] == c], N_PER_CLASS_INIT, replace=False) for c in range(10)]))
    pl = np.setdiff1d(in_tr, ini)
    assert not set(in_val) & (set(ini) | set(pl)) and not (set(in_val) | set(in_tr)) & set(test_idx)
    for s in BONUS_STRATS:
        r = run(s, sd, initial=ini, pool=pl, eval_idx=in_val)
        c = {'n_labels': [t['n_labels'] for t in r['trace']], 'accuracy': [t['accuracy'] for t in r['trace']]}
        a = auc_stats(c)
        bonus[s]['auc'].append(a['auc_norm']); bonus[s]['final'].append(a['final_accuracy'])
        bonus[s]['dist'].append(np.mean([t['batch_mean_pairwise_dist'] for t in r['trace'][1:]]))
        bonus[s]['ncls'].append(np.mean([t['batch_n_classes'] for t in r['trace'][1:]]))
mean = lambda s, k: float(np.mean(bonus[s][k])); std = lambda s, k: float(np.std(bonus[s][k], ddof=1))
print(f"{'стратегия':9s}{'AUC inner-val':>16s}{'финал':>9s}{'попарн.расст.':>15s}{'классов/батч':>14s}")
for s in BONUS_STRATS:
    print(f"{s:9s}{mean(s,'auc'):9.4f}±{std(s,'auc'):.3f}{mean(s,'final'):9.4f}{mean(s,'dist'):15.3f}{mean(s,'ncls'):14.2f}")
h1 = mean('entropy', 'dist') < min(mean('kcenter', 'dist'), mean('random', 'dist')) and mean('entropy', 'ncls') < min(mean('kcenter', 'ncls'), mean('random', 'ncls'))
h2 = mean('hybrid', 'auc') >= mean('entropy', 'auc')
main_order = comparison['ranking_by_auc_main_run']
inner_order = sorted(STRATEGIES, key=lambda s: -mean(s, 'auc'))
bonus_md = f"""# Бонус: избыточность batch-entropy sampling

## Гипотеза
H1: batch-режим entropy-сэмплинга выбирает избыточные объекты (малое попарное расстояние и малое число классов в батче) по сравнению с k-center и random.
H2: гибрид «top-{5}×k по энтропии → k-center» не хуже чистого entropy по AUC.

## Протокол
- Данные: {N_INNER} независимых стратифицированных внутренних разбиений обучающей части (70% inner-train / 30% inner-val, seeds 1000..{1000+N_INNER-1}); отложенный test не использовался ни для выбора гипотезы, ни для проверки.
- Классификатор, батч ({BATCH}), число раундов ({ROUNDS}) и initial (по {N_PER_CLASS_INIT}/класс) те же, что в основном эксперименте.
- Метрики: нормированный AUC accuracy на inner-val; среднее попарное расстояние внутри батча; число различных классов в батче (метки открыты oracle после выбора).

## Наблюдения (среднее по {N_INNER} разбиениям)
| стратегия | AUC inner-val | финальная acc | попарное расстояние в батче | классов в батче |
|---|---|---|---|---|
""" + "\n".join(f"| {s} | {mean(s,'auc'):.4f} ± {std(s,'auc'):.4f} | {mean(s,'final'):.4f} | {mean(s,'dist'):.3f} | {mean(s,'ncls'):.2f} |" for s in BONUS_STRATS) + f"""

## Сопоставление с основным экспериментом
- Порядок стратегий по AUC в основном эксперименте (test): {' > '.join(main_order)}; на inner-val: {' > '.join(inner_order)}.
- Основной эксперимент: случайный разброс random по {random_summary['n_repeats']} seeds составляет ±{random_summary['auc_norm_std']:.4f} AUC; различия меньше этого порядка нельзя считать значимыми.

## Вывод
- H1 {'подтверждена' if h1 else 'НЕ подтверждена'}: entropy даёт батч с расстоянием {mean('entropy','dist'):.3f} и {mean('entropy','ncls'):.2f} классами против {mean('kcenter','dist'):.3f}/{mean('kcenter','ncls'):.2f} (k-center) и {mean('random','dist'):.3f}/{mean('random','ncls'):.2f} (random).
- H2 {'подтверждена' if h2 else 'НЕ подтверждена'}: AUC гибрида {mean('hybrid','auc'):.4f} против {mean('entropy','auc'):.4f} у entropy (разница {mean('hybrid','auc')-mean('entropy','auc'):+.4f}).
- Ограничения: малый размер выборки ({len(train_idx)} объектов в train, {N_INNER} разбиений), простые 8×8 изображения и линейная модель; расхождения между inner-val и test объясняются другим набором оценочных объектов и случайностью initial.
"""
(ARTIFACTS / 'bonus.md').write_text(bonus_md)
save_json('bonus_raw.json', bonus)
print(bonus_md)

стратегия   AUC inner-val    финал  попарн.расст.  классов/батч
random      0.8181±0.046   0.8862          3.008          6.65
entropy     0.8217±0.021   0.9058          2.755          4.90
kcenter     0.8565±0.007   0.9222          3.205          6.28
hybrid      0.8580±0.022   0.9206          3.013          6.25
# Бонус: избыточность batch-entropy sampling

## Гипотеза
H1: batch-режим entropy-сэмплинга выбирает избыточные объекты (малое попарное расстояние и малое число классов в батче) по сравнению с k-center и random.
H2: гибрид «top-5×k по энтропии → k-center» не хуже чистого entropy по AUC.

## Протокол
- Данные: 5 независимых стратифицированных внутренних разбиений обучающей части (70% inner-train / 30% inner-val, seeds 1000..1004); отложенный test не использовался ни для выбора гипотезы, ни для проверки.
- Классификатор, батч (10), число раундов (8) и initial (по 2/класс) те же, что в основном эксперименте.
- Метрики: нормированный AUC accuracy на inner-val; среднее попарное ра

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.

### Итоговый анализ (по сохранённым артефактам, полный режим: 8 раундов, 20 → 100 меток, test = 540 изображений)

**Что измерено.** При одинаковом бюджете (20 начальных + 8 батчей по 10) нормированный AUC (средняя accuracy по бюджету) / финальная accuracy: k-center **0.862 / 0.919**, random **0.819 / 0.882** (среднее по 10 seed: 0.816±0.011 / 0.876±0.013), entropy **0.705 / 0.863**.

**Выводы.**
1. **k-center лучше random** по AUC на +0.046 (≈4σ разброса random) и на всех 10 разных начальных множествах (10/10 побед по AUC; в среднем 0.858±0.009 против 0.831±0.024, финал 0.910 против 0.883). Это самый надёжный результат работы.
2. **Entropy хуже random по AUC** (−0.11, ≈10σ; 0 побед из 10 initial-сидов). В конце бюджета отставание почти исчезает (0.863 против 0.876±0.013, т.е. в пределах ~1σ). Причина — cold start: при 20 метках модель плохо откалибрована, и топ-10 по энтропии — это 10 похожих «трудных» объектов (в бонусном эксперименте в батче entropy в среднем 4.9 класса против 6.3–6.7 у k-center/random, меньше попарное расстояние). Модель получает избыточную информацию об одной границе классов, а остальные классы остаются без новых меток.
3. **Разбор ошибок** (`test_errors.json`): у всех стратегий доминируют пары 8↔1, 9↔8, 1↔8; у k-center ошибок меньше (44 против 64 у random и 74 у entropy из 540). Entropy дополнительно путает 6→8 и 1→2, то есть хуже покрывает отдельные классы.
4. **Бонус** (inner-val внутри train, test не использовался): H1 подтверждена (entropy-батчи менее разнообразны), H2 формально подтверждена, но гибрид и k-center практически равны (0.858 против 0.857 AUC). На inner-val entropy немного лучше random (0.822 против 0.818), расхождение с основным экспериментом лежит внутри шума random на 5 разбиениях (±0.046).

**Ограничения.** Один тип данных (8×8 цифры) и линейная модель; один отложенный test из 540 объектов (стандартная ошибка accuracy ≈1.3 п.п., поэтому различия меньше ~2–3 п.п. в финальной accuracy нельзя считать значимыми); batch-размер 10 и 20 начальных меток — именно в этом режиме entropy страдает от cold start, на больших бюджетах или при калиброванной модели результат может быть иным. Oracle симулируется истинными метками, поэтому шум разметки человеком не моделируется.

In [13]:
import sys, platform, sklearn, scipy, matplotlib
save_json('environment.json', {
    'python': platform.python_version(), 'numpy': np.__version__, 'scikit-learn': sklearn.__version__,
    'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__,
    'config': {'SEED': SEED, 'FAST_DEV_RUN': FAST_DEV_RUN, 'ROUNDS': ROUNDS, 'BATCH': BATCH,
               'N_PER_CLASS_INIT': N_PER_CLASS_INIT, 'test_size': 0.3, 'N_REP': N_REP, 'N_INNER': N_INNER},
    'rerun': 'Kernel -> Restart & Run All; FAST_DEV_RUN=True для проверки кода (2 раунда), False - полный эксперимент (8 раундов)'})
print(sorted(p.name for p in ARTIFACTS.iterdir()))

['auc.json', 'bonus.md', 'bonus_raw.json', 'comparison.json', 'environment.json', 'initial_pool.json', 'learning_curves.json', 'learning_curves.png', 'multi_init.json', 'random_seeds.json', 'run_example.json', 'run_trace_entropy.json', 'run_trace_kcenter.json', 'run_trace_random.json', 'selections_demo.json', 'split.json', 'split_indices.json', 'test_errors.json', 'test_errors.png']
